# optimizer-state-tensor-buffers — worked example 1: Allocate EMA buffer matching each param's shape and device

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-state-tensor-buffers`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Stateful optimizers like Adam and RMSprop maintain a running average tensor (buffer) for every parameter. These buffers must match their corresponding parameter in shape, dtype, and device — a `(256, 768)` float32 GPU parameter needs a `(256, 768)` float32 GPU buffer. Using `t.zeros_like(p)` handles all three attributes in one call and returns a buffer that does not track gradients.

## Worked solution

**Step 1 — Materialize the parameter list.**
We call `self.params = list(params)` so the list can be iterated multiple times — once when allocating buffers, and once per `.step()` call.

**Step 2 — Allocate buffers with `zeros_like`.**
`self.ema = [t.zeros_like(p) for p in self.params]` creates one zero-initialized tensor per parameter. `zeros_like` automatically copies shape, dtype, and device from the corresponding parameter.

**Step 3 — Verify shape and dtype correspondence.**
For each `(p, buf)` pair, `p.shape == buf.shape` and `p.dtype == buf.dtype`. We also confirm `buf.requires_grad is False` — buffers are pure state, not trainable.

**Step 4 — Verify buffers are independent tensors.**
Each buffer is a distinct tensor object. Mutating one does not affect any other.

In [ ]:
import torch as t
import torch.nn as nn

class EMABuffer:
    def __init__(self, params):
        self.params = list(params)
        self.ema = [t.zeros_like(p) for p in self.params]

    def update(self, alpha=0.9):
        """Update EMA buffers: ema = alpha * ema + (1 - alpha) * param."""
        with t.no_grad():
            for p, buf in zip(self.params, self.ema):
                buf.mul_(alpha).add_((1 - alpha) * p)

# --- exercise it ---
t.manual_seed(0)
model = nn.Sequential(nn.Linear(8, 4), nn.Linear(4, 2))
ebuf = EMABuffer(model.parameters())

for (p, buf) in zip(ebuf.params, ebuf.ema):
    print(f'param {tuple(p.shape)} -> buf {tuple(buf.shape)}, '
          f'dtype_match={p.dtype==buf.dtype}, requires_grad={buf.requires_grad}')
    assert p.shape == buf.shape
    assert p.dtype == buf.dtype
    assert buf.requires_grad is False

# Run 3 EMA updates
for _ in range(3):
    ebuf.update(alpha=0.9)

print('Buffers updated, sample value:', ebuf.ema[0].abs().max().item())